In [1]:
print("GSE283655 project notebook is working")

GSE283655 project notebook is working


# GSE283655 project

Study topic: non-additive effects between interferon and MITF.

Goal: retrieve the public data

In [2]:
from pathlib import Path
from urllib.request import urlretrieve

# Folder where unchanged downloaded data will be kept
raw_dir = Path("data/raw")
raw_dir.mkdir(parents=True, exist_ok=True)

url = (
    "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE283nnn/"
    "GSE283655/suppl/GSE283655_DESeq2_TPM_values.tsv.gz"
)

output_file = raw_dir / "GSE283655_DESeq2_TPM_values.tsv.gz"

urlretrieve(url, output_file)

print("Downloaded:")
print(output_file)
print(f"File size: {output_file.stat().st_size / 1_000_000:.2f} MB")

Downloaded:
data/raw/GSE283655_DESeq2_TPM_values.tsv.gz
File size: 2.49 MB


In [3]:
import pandas as pd

file_path = "data/raw/GSE283655_DESeq2_TPM_values.tsv.gz"

expression_data = pd.read_csv(
    file_path,
    sep="\t",
    compression="gzip"
)

print("Rows and columns:", expression_data.shape)
print("\nColumn names:")
print(expression_data.columns.tolist())

expression_data.head()

Rows and columns: (40173, 13)

Column names:
['Unnamed: 0', 'siCTRL_with_IFN_1', 'siCTRL_with_IFN_2', 'siCTRL_with_IFN3', 'siCTRL_wo_IFN_1', 'siCTRL_wo_IFN_2', 'siCTRL_wo_IFN3', 'siMITF_with_IFN_1', 'siMITF_with_IFN_2', 'siMITF_with_IFN_3', 'siMITF_wo_IFN_1', 'siMITF_wo_IFN_2', 'siMITF_wo_IFN_3']


,Unnamed: 0,siCTRL_with_IFN_1,siCTRL_with_IFN_2,siCTRL_with_IFN3,siCTRL_wo_IFN_1,siCTRL_wo_IFN_2,siCTRL_wo_IFN3,siMITF_with_IFN_1,siMITF_with_IFN_2,siMITF_with_IFN_3,siMITF_wo_IFN_1,siMITF_wo_IFN_2,siMITF_wo_IFN_3
0,ENSG00000000003,22.093500,23.528791,22.810767,24.864996,25.564735,23.629445,24.238436,23.638828,22.112727,23.996147,24.042055,22.870334
1,ENSG00000000005,0.000000,0.013469,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,ENSG00000000419,142.765563,149.453618,143.782572,148.830008,168.172318,147.513155,167.209916,156.673309,148.479876,162.684370,159.665244,151.446081
3,ENSG00000000457,8.066076,11.329110,9.625513,6.434201,8.031568,6.394877,10.501443,14.334742,11.832608,9.285772,9.419900,8.947741
4,ENSG00000000460,28.176793,38.091309,32.939327,37.072441,44.795452,42.553495,19.419271,26.316202,22.274693,30.534138,33.307114,35.037533


## Download SRA run metadata

Here I download the SRA run information for GSE283655.  
This metadata identifies the 12 sequencing runs, I started with one sample

In [3]:
import pandas as pd

url = (
    "https://www.ebi.ac.uk/ena/portal/api/filereport"
    "?accession=SRR31631251"
    "&result=read_run"
    "&fields=run_accession,fastq_ftp,fastq_md5"
    "&format=tsv"
)

ena = pd.read_csv(url, sep="\t")
ena

,run_accession,fastq_ftp,fastq_md5
0,SRR31631251,ftp.sra.ebi.ac.uk/vol1/fastq/SRR316/051/SRR316...,2acdad9ecded340acb042f532500620b;31ee86a0899fc...


In [4]:
fastq_urls = ena.loc[0, "fastq_ftp"].split(";")

for url in fastq_urls:
    print(url)

ftp.sra.ebi.ac.uk/vol1/fastq/SRR316/051/SRR31631251/SRR31631251_1.fastq.gz
ftp.sra.ebi.ac.uk/vol1/fastq/SRR316/051/SRR31631251/SRR31631251_2.fastq.gz


In [ ]:
import os
import urllib.request

os.makedirs("data/raw", exist_ok=True)

for url in fastq_urls:
    full_url = "https://" + url
    filename = os.path.basename(url)
    output_path = os.path.join("data/raw", filename)

    print(f"Downloading {filename}...")
    urllib.request.urlretrieve(full_url, output_path)

print("Done.")

cleaning with fastp

In [3]:
%%bash

fastp \
  --in1 data/raw/SRR31631251_1.fastq.gz \
  --in2 data/raw/SRR31631251_2.fastq.gz \
  --out1 data/processed/SRR31631251_1.clean.fastq.gz \
  --out2 data/processed/SRR31631251_2.clean.fastq.gz \
  --json results/SRR31631251_fastp.json \
  --html results/SRR31631251_fastp.html \
  2> results/SRR31631251_fastp.log

Expression quantification using kallisto

In [4]:
%%bash
mkdir -p data/reference
cd data/reference

URL="https://ftp.ensembl.org/pub/current_fasta/homo_sapiens/cdna/Homo_sapiens.GRCh38.cdna.all.fa.gz"

curl -L "$URL" -o human_transcriptome.fa.gz

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  175M  100  175M    0     0  4612k      0  0:00:38  0:00:38 --:--:-- 4003k
